# Assignment 3

The data you will be using is the FOIA data gathered from the city of Ann Arbor on parking tickets. The repository is
made up of a set of Excel files which you're going to have to figure out how to load into a pandas `DataFrame` (time to
check the APIs!). The files may have more than one sheet in them (who knows why!?). I'd like you to answer the following
questions for me.

This is a substantial amount of data. Therefore, things will take a while to run. For testing purposes, I would
recommend using a reasonable representative subset before applying your functions on the entire dataframe. Make sure you
remove any extraneous outputs before you turn in your final copy though!


## Question 1: load_ticket_data()

First, write the code to create a single `DataFrame` object in a function called `load_ticket_data()`. This function
should return the full dataframe and take no parameters (you can assume the ticket files are in the same directory as
your assignment notebook). Column labels should be as follows:

`['Ticket #', 'Badge', 'Issue Date', 'IssueTime', 'Plate', 'State', 'Make', 'Model', 'Violation', 'Description', 'Location', 'Meter', 'Fine', 'Penalty']`

Here are some hints.

<ol>
<li> Be sure to scroll through every single sheet to make sure what rows should be dropped.</li>
<li> Make sure to exclude unnecessary footers and headers from the datafile.</li>
<li> Check if your header labels are correct.</li>
</ol>


In [9]:
def load_ticket_data():
    import xlrd
    import pandas as pd
    import numpy as np
    import re
    from mads.lib.path import assets

    # Filter all warnings. If you would like to see the warnings, please comment the two lines below.
    import warnings

    warnings.filterwarnings("ignore")

    # YOUR CODE HERE
    headers = ['Ticket #', 'Badge', 'Issue Date', 'IssueTime', 'Plate', 'State', 'Make', 'Model', 'Violation', 'Description', 'Location', 'Meter', 'Fine', 'Penalty']

    #Ticket violations 2015
    sheets2015 = pd.read_excel("AnnArbor-TicketViolation2015.xls", names=headers, sheet_name=None)
    sheets2016 = pd.read_excel("AnnArbor-TicketViolation2016.xls", names=headers, sheet_name=None)
    sheets2017 = pd.read_excel("AnnArbor-TicketViolation2017.xls", names=headers, sheet_name=None)
    sheets2018 = pd.read_excel("AnnArbor-TicketViolation2018.xls", names=headers, sheet_name=None)
    sheets2019 = pd.read_excel("AnnArbor-TicketViolation2019.xls", names=headers, sheet_name=None)
    sheets2020 = pd.read_excel("AnnArbor-TicketViolation-jan2020.xls", names=headers, sheet_name=None)

    all_sheets = [sheets2015, sheets2016, sheets2017, sheets2018, sheets2019, sheets2020]
    dataset = []
    
    for n, dct in enumerate(all_sheets):
        middle_sheets = []
        i = 0
        #print(n)
        sheet_count = len(dct)
        #print("file:")
        if sheet_count == 1:
            df = next(iter(dct.values()))
            df = df.iloc[1:-1]
            dataset.append(df)
        else:    
            for sheet_name, df in dct.items():
                #print(sheet_name)
                if i == 0:
                    sheet1 = df.iloc[1:]
                    #print("Length:  ", len(sheet1))
                elif i == sheet_count - 1:
                    last_sheet = df.iloc[:-1]
                else:
                    middle_sheets.append(df)
                i += 1
            file_df = pd.concat([sheet1] + middle_sheets + [last_sheet],ignore_index=True)
            dataset.append(file_df)
        
    final_df = pd.concat(dataset, ignore_index=True)  
    #final_df = final_df[final_df["Violation"].str.strip() != ""]
    #828314
    #final_df = final_df[final_df["Violation"].notna()]
    #final_df = final_df[final_df["Fine"] > 0]
    #df12015 = sheets2015["Sheet1"].iloc[1:]
    #df22015 = sheets2015["Sheet2"]
    #df32015 = sheets2015["Sheet3"].iloc[:-1]

    print("Length:  ", len(final_df))
    
    #print(df12015.head(2))
    #print(df22015.head(2))
    #print(df32015.tail(2))
    return final_df

In [10]:
#load_ticket_data()

In [11]:
import xlrd
import pandas as pd
import numpy as np
import re

df_1_test = load_ticket_data()
assert isinstance(
    df_1_test, pd.DataFrame
), "Q1: What your function returns must be pd.DataFrame."
assert len(df_1_test) == 811439, "Q1: There should be 811439 rows in the dataframe."
assert len(df_1_test.columns) == 14, "Q1: There should be 14 columns in the dataframe."

Length:   811439


## Question 2: generate_descriptors()

Write a function called `generate_descriptors(df)` which takes in the DataFrame you loaded from Question1 and returns a
dataframe of all unique ticket descriptions and how frequent they are (e.g. it will tell you how many "HANDICAP" or "NO
PERMITS U/M" tickets have been issued) for each of the following three time periods: morning (3 am to 11:59 am),
afternoon (12 pm - 5:59 pm), and evening (6 pm - 2:59 am).

- Make sure you drop na values of input `df`.
- The DataFrame which `generate_descriptors(df)` returns should have 3 rows and 51 columns.
- Index should be labelled as `Morning`, `Afternoon`, and `Evening`.
- Column names should be unique values of `df["Description"]`.


In [52]:
df = load_ticket_data()


Length:   811439


In [53]:
#df["Description"].unique()
#len(df["Description"].unique())

In [54]:
#df.head()

In [55]:
#type(df["IssueTime"][2])

In [12]:
def time_buckets(time):
    time = int(time)
    if 300 <= time <= time <= 1159:
        return "Morning"
    elif 1200 <= time <= 1759:
        return "Afternoon"
    else:
        return "Evening"    


In [13]:
#df["TOD"] = df["IssueTime"].apply(time_buckets)

In [14]:
#df.head()

In [15]:
def generate_descriptors(df):
    import xlrd
    import pandas as pd
    import numpy as np
    import re

    # Filter all warnings. If you would like to see the warnings, please comment the two lines below.
    import warnings

    warnings.filterwarnings("ignore")

    # YOUR CODE HERE
    df["TOD"] = df["IssueTime"].apply(time_buckets)

    df = df.dropna(subset=["Description"])

    df_q2 = pd.crosstab(
        df["TOD"],
        df["Description"]
    )

    #new_df = df_clean.pivot_table(
    #    values="IssueTime",
    #    index="TOD",
    #    columns="Description",
    #    aggfunc="count"
    #)

    df_q2 = df_q2.reindex(["Morning", "Afternoon", "Evening"])

    df_q2.index.name = None
    
    
    return df_q2

In [16]:
df_q2 = generate_descriptors(load_ticket_data())
assert df_q2.shape == (3, 51), "Q2: The shape of the DataFrame is incorrect."
assert "Morning" in df_q2.index, 'Q2: "Morning" shoud be in the index of the DataFrame'
assert (
    "Afternoon" in df_q2.index
), 'Q2: "Afternoon" should be in the index of the DataFrame'
assert "Evening" in df_q2.index, 'Q2: "Evening" should be in the index of the DataFrame'

Length:   811439


## Question 3: common_car_make()

What is the most common make of car which received tickets from the state of NY? The answer should be a string.


In [17]:
df = load_ticket_data()
df.head()

Length:   811439


,Ticket #,Badge,Issue Date,IssueTime,Plate,State,Make,Model,Violation,Description,Location,Meter,Fine,Penalty
0,H000210594,036,2015-01-01 00:00:00,2214,LAS5658,OH,SUBA,NaN,A04,NO PRKNG ANYTME,525 ELM,NaN,35,20
1,2100005782,821,2015-01-02 00:00:00,0824,DEZ4465,MI,FORD,NaN,A01,EXPIRED METER,600 BLK OF STATE SOU,4006A,10,0
2,2110008524,826,2015-01-02 00:00:00,1719,DCM1327,MI,SATU,NaN,A01,EXPIRED METER,FARMER'S MARKET,17,10,0
3,2110008525,826,2015-01-02 00:00:00,1725,BAX385,IA,CHEV,NaN,A01,EXPIRED METER,FARMER'S MARKET,35,10,0
4,2100005834,821,2015-01-02 00:00:00,1344,2LEH1,MI,FORD,NaN,A04,NO PRKNG ANYTME,600 BLK OF WILLIAM E,NaN,25,0


In [18]:
def common_car_make():
    import xlrd
    import pandas as pd
    import numpy as np
    import re

    # Filter all warnings. If you would like to see the warnings, please comment the two lines below.
    import warnings

    warnings.filterwarnings("ignore")

    df = load_ticket_data()
    # YOUR CODE HERE

    answer = (
        df[df["State"] == "NY"]
        .groupby("Make")
        .size()
        .idxmax()
    )
    
    return answer


    

In [19]:
answer3 = common_car_make()
assert isinstance(answer3, str), "Q3: Your answer should be a string type."

Length:   811439


## Question 4: fine_per_plates()

Starting in 2004 Michigan moved to issuing plates with the format of ABC1234. That got me thinking, how many vanity
plate holders there are in our dataset? Count for me the number of Michigan vehicles with plates in the following
formats that have received a ticket:

- ABC1234
- ABC123
- 123ABC
- Vanity Plates (i.e. anything other than the aforementioned formats, including missing or NaN values)

Complete the function `fine_per_plates()` returning a dictionary. The dictinary should be formatted as follows:

```
plates_dict = {"ABC1234":the_number_of_vehicles,
                "ABC123":the_number_of_vehicles,
                "123ABC":the_number_of_vehicles,
                "vanity":the_number_of_vehicles}
```


In [45]:
def plate_type(plate):

    #plate = plate.fillna("NONE")
    plate = str(plate).strip().upper()
    pattern1 = r"^[A-Z]{3}[0-9]{4}$"
    pattern2 = r"^[A-Z]{3}[0-9]{3}$"
    pattern3 = r"^[0-9]{3}[A-Z]{3}$"
    
    if re.search(pattern1, plate):
        return "ABC1234"
    elif re.search(pattern2, plate):
        return "ABC123"  
    elif re.search(pattern3, plate):
        return "123ABC"     
    else:
        return "vanity"
    


In [35]:
df = load_ticket_data()

Length:   811439


In [49]:
df["Plate Type"] = df["Plate"].apply(plate_type)


counts_df = (
    df.groupby("Plate Type")
    .size()
)

plates_dict = {"ABC1234": counts_df["ABC1234"],
                "ABC123": counts_df["ABC123"],
                "123ABC": counts_df["123ABC"],
                "vanity": counts_df["vanity"]
}

plates_dict
#{'ABC1234': 511908, 'ABC123': 48459, '123ABC': 12310, 'vanity': 238762}
#plates_dict

{'ABC1234': 511908, 'ABC123': 48459, '123ABC': 12310, 'vanity': 238762}

In [44]:
print(511908 + 48459 + 12310 + 238762)

811439


In [52]:
def fine_per_plates():
    import xlrd
    import pandas as pd
    import numpy as np
    import re

    # Filter all warnings. If you would like to see the warnings, please comment the two lines below.
    import warnings

    warnings.filterwarnings("ignore")

    df = load_ticket_data()
    # YOUR CODE HERE
    df = df[df["State"] == "MI"]
    
    df["Plate Type"] = df["Plate"].apply(plate_type)

    counts_df = (
        df.groupby("Plate Type")
        .size()
    )

    plates_dict = {"ABC1234": counts_df["ABC1234"],
                "ABC123": counts_df["ABC123"],
                "123ABC": counts_df["123ABC"],
                "vanity": counts_df["vanity"]
    }

    return plates_dict

    

In [51]:
assert len(fine_per_plates()) == 4, "Return a dictionary with four items."



Length:   811439


In [48]:
fine_per_plates()

Length:   811439


{'ABC1234': 511908, 'ABC123': 48459, '123ABC': 12310, 'vanity': 238762}